# Chạy giao diện FastAPI + React hỏi đáp pháp luật trên Kaggle

Notebook này cài giao diện trong `demo/web/` và `src/drill_ir/demo/`, đọc `OPENAI_API_KEY` và `NGROK_AUTH_TOKEN` từ Kaggle Secrets, gọi Chat Completions qua OpenAI-compatible endpoint, rồi mở ứng dụng qua Ngrok. Không ghi secret ra output.

> **Đầu vào Kaggle:** corpus `legal-corpus.parquet`; model VietLegal-Harrier và multilingual-E5-large; output notebook fine-tune có checkpoint-1234; output notebook baseline có archive `*_complete_results.tar.gz` (LightGBM + Multiclass–Ridge). Cache BM25/dense là tùy chọn; thiếu cache thì notebook tự dựng index và lần khởi động đầu có thể lâu. Chọn GPU T4 trong phần Settings của notebook.

Bật Internet trong Kaggle để clone repo, tải tokenizer và cài package. Tạo ba Kaggle Secrets riêng: `GITHUB_TOKEN`, `OPENAI_API_KEY`, `NGROK_AUTH_TOKEN`.
Frontend được build bằng Node.js, FastAPI phục vụ cả giao diện và API qua một cổng Ngrok. Chỉ dùng một worker để không nhân đôi model/GPU memory. Notebook tự nạp Node.js 22.16.0 nếu image Kaggle chưa có runtime tương thích.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time
import urllib.request
from kaggle_secrets import UserSecretsClient

REPO_URL = 'https://github.com/leeminsun1205/Drill-reliable-retrieval.git'
REPO_DIR = Path('/kaggle/working/Drill-reliable-retrieval')
APP_DIR = REPO_DIR
WEB_DIR = REPO_DIR / 'demo' / 'web'
PORT = 8501
OPENAI_MODEL = 'gpt-4o-mini'
OPENAI_BASE_URL = 'https://llm.wokushop.com/v1'
NGROK_REGION = 'ap'
RETRIEVER_FACTORY = 'drill_ir.demo.pipeline_adapter:create_backend'

github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Thiếu Kaggle Secret GITHUB_TOKEN để clone repository')
if not REPO_DIR.exists():
    askpass = Path('/kaggle/working/.drill-git-askpass.sh')
    askpass.write_text('#!/bin/sh\ncase \"$1\" in\n  *Username*) printf \"%s\\n\" x-access-token ;;\n  *Password*) printf \"%s\\n\" \"$GITHUB_TOKEN\" ;;\n  *) exit 1 ;;\nesac\n')
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True, env=git_env)
    finally:
        askpass.unlink(missing_ok=True)
if not (WEB_DIR / 'package.json').is_file():
    raise FileNotFoundError(f'Không tìm thấy frontend: {WEB_DIR}')
print({'repo': str(REPO_DIR), 'app': str(APP_DIR), 'port': PORT, 'retriever_factory_configured': bool(RETRIEVER_FACTORY)})

In [ ]:
# Cài pipeline retrieval, VnCoreNLP, LightGBM, giao diện và pyngrok.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dense,vietnamese,ltr,demo]'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyngrok>=7,<8'], check=True)
vncore_dir = Path('/kaggle/working/VnCoreNLP')
vncore_jar = vncore_dir / 'VnCoreNLP-1.2.jar'
if not vncore_jar.is_file():
    if vncore_dir.exists():
        import shutil; shutil.rmtree(vncore_dir)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vncorenlp/VnCoreNLP.git', str(vncore_dir)], check=True)
if not vncore_jar.is_file():
    raise FileNotFoundError(f'Không tìm thấy VnCoreNLP JAR: {vncore_jar}')
print('Cài đặt hoàn tất.')
# Reuse a compatible Node, otherwise install a pinned runtime in /kaggle/working.
import hashlib
import platform
import shutil
import tarfile
node_ok = False
if shutil.which('node') and shutil.which('npm'):
    version = subprocess.check_output(['node', '--version'], text=True).strip().lstrip('v')
    major, minor, *_ = map(int, version.split('.'))
    node_ok = (major == 20 and minor >= 19) or (major == 22 and minor >= 12) or major > 22
if not node_ok:
    if platform.system() != 'Linux' or platform.machine() != 'x86_64':
        raise RuntimeError('Bootstrap Node chỉ hỗ trợ Kaggle Linux x86_64.')
    node_version = '22.16.0'
    filename = f'node-v{node_version}-linux-x64.tar.xz'
    node_base = f'https://nodejs.org/dist/v{node_version}'
    archive = Path('/kaggle/working') / filename
    with urllib.request.urlopen(f'{node_base}/SHASUMS256.txt', timeout=60) as response:
        checksums = response.read().decode().splitlines()
    expected = next(line.split()[0] for line in checksums if line.split()[-1] == filename)
    urllib.request.urlretrieve(f'{node_base}/{filename}', archive)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
        raise RuntimeError('Node.js archive checksum mismatch')
    with tarfile.open(archive) as bundle:
        bundle.extractall('/kaggle/working', filter='data')
    os.environ['PATH'] = str(Path('/kaggle/working') / f'node-v{node_version}-linux-x64' / 'bin') + os.pathsep + os.environ['PATH']
subprocess.run(['npm', 'ci', '--no-audit', '--no-fund'], cwd=WEB_DIR, check=True)
subprocess.run(['npm', 'run', 'build'], cwd=WEB_DIR, check=True)


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
openai_key = secrets.get_secret('OPENAI_API_KEY')
ngrok_token = secrets.get_secret('NGROK_AUTH_TOKEN')
if not openai_key:
    raise RuntimeError('Thiếu Kaggle Secret OPENAI_API_KEY')
if not ngrok_token:
    raise RuntimeError('Thiếu Kaggle Secret NGROK_AUTH_TOKEN')
os.environ['OPENAI_API_KEY'] = openai_key
os.environ['OPENAI_MODEL'] = OPENAI_MODEL
os.environ['OPENAI_BASE_URL'] = OPENAI_BASE_URL
if RETRIEVER_FACTORY:
    os.environ['DRILL_RETRIEVER_FACTORY'] = RETRIEVER_FACTORY
print('Đã nạp secrets vào môi trường tiến trình; không hiển thị giá trị.')

In [ ]:
# Kiểm tra các input và model artifact trước khi mở public tunnel.
sys.path.insert(0, str(APP_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))  # Editable install runs in a child pip process.
from drill_ir.demo import pipeline_adapter as adapter
corpus_path = Path(os.environ.get('DRILL_CORPUS_PATH') or adapter._unique_file('legal-corpus.parquet'))
archive_path = Path(os.environ.get('DRILL_COMPLETE_RESULTS_ARCHIVE') or adapter._unique_file(adapter.ARTIFACT_GLOB))
checkpoint_path = Path(os.environ.get('DRILL_CHECKPOINT_PATH') or adapter._checkpoint_root())
harrier_path = Path(os.environ.get('DRILL_HARRIER_PATH') or adapter._model_root('vietlegal-harrier'))
e5_path = Path(os.environ.get('DRILL_E5_PATH') or adapter._model_root('multilingual-e5-large'))
vncore_jar = Path('/kaggle/working/VnCoreNLP/VnCoreNLP-1.2.jar')
if not corpus_path.is_file() or not vncore_jar.is_file():
    raise FileNotFoundError(f'Corpus or VnCoreNLP JAR missing: corpus={corpus_path}, jar={vncore_jar}')
extraction = adapter.WORK_ROOT / 'complete_results'
preflight_artifacts = adapter._load_artifacts(archive_path, extraction, checkpoint_path)
cache_hits = {name: hit for name, (_path, hit) in {
    'bm25': adapter._resolve_retrieval_cache('bm25s_vncorenlp'),
    'chunks': adapter._resolve_retrieval_cache('bge_m3_token_window448_chunks'),
    'harrier': adapter._resolve_retrieval_cache('vietlegal_harrier_0_6b_token_window448'),
    'e5': adapter._resolve_retrieval_cache('multilingual_e5_large_token_window448'),
}.items()}
print({'corpus': str(corpus_path), 'checkpoint': str(checkpoint_path), 'harrier': str(harrier_path), 'e5': str(e5_path), 'results_archive': str(archive_path), 'feature_count': len(preflight_artifacts.lightgbm_feature_names), 'cache_hits': cache_hits})

In [ ]:
from pyngrok import conf, ngrok

conf.get_default().auth_token = ngrok_token
conf.get_default().region = NGROK_REGION
# Dọn tunnel cũ trong cùng phiên trước khi mở tunnel mới.
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)
if 'api_process' in globals() and api_process.poll() is None:
    api_process.terminate()
    try:
        api_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        api_process.kill()
if 'log_file' in globals() and not log_file.closed:
    log_file.close()

log_path = Path('/kaggle/working/legal_qa_api.log')
log_file = log_path.open('w', encoding='utf-8')
api_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'drill_ir.demo.api:app',
     '--host', '0.0.0.0', '--port', str(PORT), '--workers', '1'],
    cwd=APP_DIR, env=os.environ.copy(), stdout=log_file, stderr=subprocess.STDOUT,
)

health_url = f'http://127.0.0.1:{PORT}/api/health'
deadline = time.monotonic() + 1800
while time.monotonic() < deadline:
    if api_process.poll() is not None:
        log_file.flush()
        raise RuntimeError(f'FastAPI + React dừng sớm. Xem log tại {log_path}')
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            if response.status == 200:
                import json
                if not json.load(response)['ready']:
                    raise RuntimeError(f'Pipeline khởi tạo thất bại. Xem {log_path}')
                break
    except (OSError, TimeoutError):
        time.sleep(2)
else:
    raise TimeoutError(f'FastAPI + React chưa sẵn sàng. Xem log tại {log_path}')

public_url = ngrok.connect(PORT, 'http').public_url
print('FastAPI + React đang chạy:', public_url)
print('Giữ phiên notebook hoạt động để duy trì ứng dụng. Lịch sử chat chỉ nằm trong tab trình duyệt.')
if not RETRIEVER_FACTORY:
    print('Lưu ý: giao diện mở được nhưng chat đang khóa vì chưa cấu hình retrieval adapter.')

## Dừng ứng dụng

Chạy cell dưới đây khi muốn dừng FastAPI + React và đóng các tunnel Ngrok của phiên notebook.

In [ ]:
try:
    ngrok.kill()
except Exception as error:
    print(f'Không đóng được tunnel: {error}')
if 'api_process' in globals() and api_process.poll() is None:
    api_process.terminate()
    try:
        api_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        api_process.kill()
if 'log_file' in globals():
    log_file.close()
print('Đã dừng ứng dụng.')